In [24]:
import os
import requests
import json
from datetime import datetime, timezone

# API key fijada directamente
os.environ["NASA_API_KEY"] = "WAdfhAnjGGOggxiEdP9CDuKS3Py1j5v1ianvkMaQ"

# Cargar API key
API_KEY = os.getenv("NASA_API_KEY", "").strip()

if not API_KEY:
    raise ValueError(
        "No se encontró NASA_API_KEY. Define una clave válida antes de continuar."
    )

print(f" API key configurada: {API_KEY[:4]}{'*' * (len(API_KEY) - 4)}")
print("Librerías cargadas ")

 API key configurada: WAdf************************************
Librerías cargadas 


In [20]:
NOMBRE = "Abraham Vallejo Jimenez"
CARNET = "202108790"

print(f"Estudiante: {NOMBRE}")
print(f"Carnet: {CARNET}")

Estudiante: Abraham Vallejo Jimenez
Carnet: 202108790


In [14]:
# Parámetros de la solicitud
URL_BASE = "https://api.nasa.gov/DONKI/GST"
start_date = "2024-05-01"
end_date = "2024-05-31"


url = f"{URL_BASE}?startDate={start_date}&endDate={end_date}&api_key={API_KEY}"

# Ejecutar solicitud
respuesta = requests.get(url, timeout=15)

# Verificar que la solicitud
if respuesta.status_code != 200:
    print(f"Error {respuesta.status_code}")
    print(f"Mensaje: {respuesta.text}")
    raise SystemExit("No se pudo obtener los datos. Intenta de nuevo.")

tormentas = respuesta.json()

print(f" Solicitud exitosa")
print(f" {len(tormentas)} tormentas geomagnéticas recibidas")

 Solicitud exitosa
 5 tormentas geomagnéticas recibidas


In [15]:
print(f"\n{'Evento':<8} {'ID de Tormenta':<35} {'Fecha de Inicio'}")
print("-" * 70)

for i, tormenta in enumerate(tormentas, 1):
    gst_id = tormenta["gstID"]
    start_time = tormenta["startTime"]
    print(f"{i:<8} {gst_id:<35} {start_time}")


Evento   ID de Tormenta                      Fecha de Inicio
----------------------------------------------------------------------
1        2024-05-02T15:00:00-GST-001         2024-05-02T15:00Z
2        2024-05-10T15:00:00-GST-001         2024-05-10T15:00Z
3        2024-05-12T21:00:00-GST-001         2024-05-12T21:00Z
4        2024-05-16T06:00:00-GST-001         2024-05-16T06:00Z
5        2024-05-17T18:00:00-GST-001         2024-05-17T18:00Z


In [16]:
# Encontrar Kp máximo de cada tormenta
print("\n=== KP MÁXIMO POR TORMENTA ===")

kp_maximos = []

for tormenta in tormentas:
    mediciones = tormenta["allKpIndex"]


    valores_kp = [m["kpIndex"] for m in mediciones]
    kp_max = max(valores_kp)

    kp_maximos.append(kp_max)

    print(f"  {tormenta['startTime']}  →  Kp máx = {kp_max:.2f}")

# Encontrar la tormenta más intensa
print("\n=== TORMENTA MÁS INTENSA ===")

idx_max = kp_maximos.index(max(kp_maximos))
tormenta_mas_intensa = tormentas[idx_max]
kp_global_max = kp_maximos[idx_max]

print(f"ID:        {tormenta_mas_intensa['gstID']}")
print(f"Inicio:    {tormenta_mas_intensa['startTime']}")
print(f"Kp máximo: {kp_global_max:.2f}")


=== KP MÁXIMO POR TORMENTA ===
  2024-05-02T15:00Z  →  Kp máx = 6.67
  2024-05-10T15:00Z  →  Kp máx = 9.00
  2024-05-12T21:00Z  →  Kp máx = 6.33
  2024-05-16T06:00Z  →  Kp máx = 6.00
  2024-05-17T18:00Z  →  Kp máx = 6.00

=== TORMENTA MÁS INTENSA ===
ID:        2024-05-10T15:00:00-GST-001
Inicio:    2024-05-10T15:00Z
Kp máximo: 9.00


In [17]:
def clasificar_kp(kp):
    """
    Clasifica la intensidad de una tormenta geomagnética según el índice Kp.

    Args:
        kp (float): valor del índice Kp (0-9)

    Returns:
        str: categoría de la tormenta
    """
    kp_int = int(kp)  # Redondear hacia abajo

    if kp_int < 4:
        return "Quieto"
    elif kp_int == 4:
        return "Activo"
    elif kp_int == 5:
        return "G1 - Menor"
    elif kp_int == 6:
        return "G2 - Moderada"
    elif kp_int == 7:
        return "G3 - Fuerte"
    elif kp_int == 8:
        return "G4 - Severa"
    else:  # kp_int >= 9
        return "G5 - Extrema"


# Verificar la función con valores de prueba
print("Pruebas de la función clasificar_kp():")
for kp_test in [2.0, 4.0, 5.33, 6.67, 7.0, 8.67, 9.0]:
    resultado = clasificar_kp(kp_test)
    print(f"  Kp={kp_test:.2f}  →  {resultado}")

Pruebas de la función clasificar_kp():
  Kp=2.00  →  Quieto
  Kp=4.00  →  Activo
  Kp=5.33  →  G1 - Menor
  Kp=6.67  →  G2 - Moderada
  Kp=7.00  →  G3 - Fuerte
  Kp=8.67  →  G4 - Severa
  Kp=9.00  →  G5 - Extrema


In [18]:
# Tabla de clasificación de todas las tormentas
print("\n=== CLASIFICACIÓN DE TORMENTAS ===")
print(f"\n{'ID Tormenta':<35} {'Kp máx':>7}  {'Categoría'}")
print("-" * 70)

for tormenta, kp_max in zip(tormentas, kp_maximos):
    categoria = clasificar_kp(kp_max)
    print(f"{tormenta['gstID']:<35} {kp_max:>7.2f}  {categoria}")


=== CLASIFICACIÓN DE TORMENTAS ===

ID Tormenta                          Kp máx  Categoría
----------------------------------------------------------------------
2024-05-02T15:00:00-GST-001            6.67  G2 - Moderada
2024-05-10T15:00:00-GST-001            9.00  G5 - Extrema
2024-05-12T21:00:00-GST-001            6.33  G2 - Moderada
2024-05-16T06:00:00-GST-001            6.00  G2 - Moderada
2024-05-17T18:00:00-GST-001            6.00  G2 - Moderada


In [19]:
def analizar_tormenta(tormenta):
    """
    Realiza análisis estadístico completo de una tormenta geomagnética.

    Args:
        tormenta (dict): evento de tormenta del endpoint DONKI/GST

    Returns:
        dict: análisis con claves: id, inicio, kp_max, kp_min, kp_promedio,
              num_mediciones, categoria, eventos_vinculados
    """
    # Extraer lista de mediciones
    mediciones = tormenta["allKpIndex"]

    # Extraer valores de Kp
    valores_kp = [m["kpIndex"] for m in mediciones]

    # Calcular estadísticas
    kp_max = max(valores_kp)
    kp_min = min(valores_kp)
    kp_promedio = round(sum(valores_kp) / len(valores_kp), 2)
    num_mediciones = len(mediciones)

    # Clasificar según Kp máximo
    categoria = clasificar_kp(kp_max)

    # Contar eventos
    linked_events = tormenta["linkedEvents"] or []
    eventos_vinculados = len(linked_events)


    return {
        "id": tormenta["gstID"],
        "inicio": tormenta["startTime"],
        "kp_max": kp_max,
        "kp_min": kp_min,
        "kp_promedio": kp_promedio,
        "num_mediciones": num_mediciones,
        "categoria": categoria,
        "eventos_vinculados": eventos_vinculados,
    }


# Aplicar función a todas las tormentas
print("\n=== ANÁLISIS COMPLETO DE TORMENTAS ===")

analisis_lista = []

for i, tormenta in enumerate(tormentas, 1):
    analisis = analizar_tormenta(tormenta)
    analisis_lista.append(analisis)

    print(f"\n[Tormenta {i}]")
    print(f"  ID                : {analisis['id']}")
    print(f"  Inicio            : {analisis['inicio']}")
    print(f"  Kp máximo         : {analisis['kp_max']:.2f}")
    print(f"  Kp mínimo         : {analisis['kp_min']:.2f}")
    print(f"  Kp promedio       : {analisis['kp_promedio']}")
    print(f"  # Mediciones      : {analisis['num_mediciones']}")
    print(f"  Categoría         : {analisis['categoria']}")
    print(f"  Eventos vinculados: {analisis['eventos_vinculados']}")


=== ANÁLISIS COMPLETO DE TORMENTAS ===

[Tormenta 1]
  ID                : 2024-05-02T15:00:00-GST-001
  Inicio            : 2024-05-02T15:00Z
  Kp máximo         : 6.67
  Kp mínimo         : 6.67
  Kp promedio       : 6.67
  # Mediciones      : 2
  Categoría         : G2 - Moderada
  Eventos vinculados: 2

[Tormenta 2]
  ID                : 2024-05-10T15:00:00-GST-001
  Inicio            : 2024-05-10T15:00Z
  Kp máximo         : 9.00
  Kp mínimo         : 6.67
  Kp promedio       : 8.13
  # Mediciones      : 13
  Categoría         : G5 - Extrema
  Eventos vinculados: 6

[Tormenta 3]
  ID                : 2024-05-12T21:00:00-GST-001
  Inicio            : 2024-05-12T21:00Z
  Kp máximo         : 6.33
  Kp mínimo         : 5.67
  Kp promedio       : 6.0
  # Mediciones      : 3
  Categoría         : G2 - Moderada
  Eventos vinculados: 2

[Tormenta 4]
  ID                : 2024-05-16T06:00:00-GST-001
  Inicio            : 2024-05-16T06:00Z
  Kp máximo         : 6.00
  Kp mínimo         : 6

In [22]:
# Obtener posición actual de la ISS (sello de entrega)
print("Consultando posición actual de la ISS...")
try:
    r_iss = requests.get("http://api.open-notify.org/iss-now.json", timeout=8)
    r_iss.raise_for_status()
    iss_data = r_iss.json()
    iss_lat = float(iss_data["iss_position"]["latitude"])
    iss_lon = float(iss_data["iss_position"]["longitude"])
    iss_ts = iss_data["timestamp"]
    iss_hora = datetime.fromtimestamp(iss_ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    iss_ok = True
except Exception as e:
    iss_lat, iss_lon, iss_hora = 0.0, 0.0, "no disponible"
    iss_ok = False
    print(f"⚠ No se pudo obtener posición ISS ({e})")

# Timestamp de entrega
ts_entrega = datetime.now(tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

# Imprimir sello
print()
print("=" * 60)
print("               SELLO DE ENTREGA — PARCIAL 2")
print("=" * 60)
print(f"Estudiante    : {NOMBRE}")
print(f"Carnet        : {CARNET}")
print(f"Fecha/Hora    : {ts_entrega}")
print(f"ISS Latitud   : {iss_lat:+.4f}°")
print(f"ISS Longitud  : {iss_lon:+.4f}°")
print(f"ISS Hora UTC  : {iss_hora}")
print("=" * 60)

if not iss_ok:
    print(" Sello generado sin datos de ISS (sin conexión)")

Consultando posición actual de la ISS...

               SELLO DE ENTREGA — PARCIAL 2
Estudiante    : Abraham Vallejo Jimenez
Carnet        : 202108790
Fecha/Hora    : 2026-04-24 05:16:35 UTC
ISS Latitud   : +45.2252°
ISS Longitud  : +42.7887°
ISS Hora UTC  : 2026-04-24 05:16:35 UTC
